In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_excel("/content/drive/MyDrive/Telco_customer_churn.xlsx")

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['Churn Value'].value_counts()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Tenure Months'], bins= 30, kde=True)
plt.xlabel('Tenure Months')
plt.ylabel('Customer Count')
plt.title('Distribution of Tenure Months vs no.of.customers')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x = 'Churn Label', y = 'Tenure Months', data = df)
plt.xlabel('Churn Label')
plt.ylabel('Tenure Months')
plt.title('Churn Label vs Tenure Months')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Monthly Charges'], bins= 30, kde=True)
plt.xlabel('Monthly Charges')
plt.ylabel('Customer Count')
plt.title('Distribution of Monthly Charges vs no.of.customers')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x = 'Churn Label', y = 'Monthly Charges', data = df)
plt.xlabel('Churn Label')
plt.ylabel('Monthly Charges')
plt.title('Churn Label vs Monthly Charges')
plt.show()

In [ ]:
df[df["Churn Label"] == 'Yes']["Monthly Charges"].quantile([0.25,0.5,0.75])

In [ ]:
df[df["Churn Label"] == 'No']["Monthly Charges"].quantile([0.25,0.5,0.75])

In [ ]:
df["Monthly Charges"].describe()

In [ ]:
df['Contract'].unique()

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x = 'Contract', hue = 'Churn Label', data = df)
plt.xlabel('Contract')
plt.ylabel('Customer Count')
plt.title('Contract vs Customer Count')
plt.show()

In [ ]:
df['Internet Service'].unique()

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x = 'Internet Service', hue = 'Churn Label', data = df)
plt.xlabel('Internet Service')
plt.ylabel('Customer Count')
plt.title('Internet Service vs Customer Count')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x = 'Payment Method', hue = 'Churn Label', data = df)
plt.xlabel('Payment Method')
plt.ylabel('Customer Count')
plt.xticks(rotation = 45)
plt.title('Payment Method vs Customer Count')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x = 'Tech Support', hue = 'Churn Label', data = df)
plt.xlabel('Tech Support')
plt.ylabel('Customer Count')
plt.title('Tech Support vs Customer Count')
plt.show()

In [ ]:
avg_tenure = df.groupby(["Churn Label"])["Tenure Months"].mean()
avg_tenure

In [ ]:
numerical_columns = ['Tenure Months','Monthly Charges', 'Churn Value', 'Churn Score', 'CLTV']
correlation_matrix = df[numerical_columns].corr()
correlation_matrix

In [ ]:
pd.crosstab(df['Contract'], df['Churn Label'], normalize= 'index')

**Data Cleaning**

In [ ]:
df['Total Charges'] =  pd.to_numeric(df['Total Charges'], errors='coerce')

In [ ]:
df['Total Charges'].dtype

In [ ]:
df['Total Charges'].isnull().sum()

In [ ]:
df['Tenure Months'][df['Total Charges'].isnull()]

In [ ]:
df['Total Charges'] = df['Total Charges'].fillna(0)

In [ ]:
drop_columns = ['CustomerID', 'Count', 'Country', 'State', 'Lat Long', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Churn Score', 'CLTV', 'Churn Reason', 'Churn Score']
df.drop(columns = drop_columns, inplace = True)


In [ ]:
df.shape

In [ ]:
df_encoded = pd.get_dummies(data = df, drop_first= True)
df_encoded.shape

In [ ]:
df = df.drop(columns= ["City"])

In [ ]:
df.drop(columns= ["Churn Label"], inplace = True)

In [ ]:
df_encoded = pd.get_dummies(data = df, drop_first= True)
df_encoded.shape

In [ ]:
x = df_encoded.drop(columns= ["Churn Value"])
y = df_encoded["Churn Value"]

Machine Learning Implementation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print(f"Random Forest Classifier Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)
report = classification_report(y_test, y_pred_rf)
print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(report)

Approach - 1 : Handle class Imbalance

In [ ]:
rf_balanced = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
rf_balanced.fit(X_train, y_train)
y_pred_rf_balanced = rf_balanced.predict(X_test)
print(f"Random Forest Classifier (Balanced) Accuracy: {accuracy_score(y_test, y_pred_rf_balanced):.4f}")
print("\nClassification Report (Balanced):")
print(classification_report(y_test, y_pred_rf_balanced))
print("\nConfusion Matrix (Balanced):")
print(confusion_matrix(y_test, y_pred_rf_balanced))

Approach - 2 : Hyperparameter tuning

In [ ]:
rf_tuned= RandomForestClassifier(n_estimators=300, max_depth = 10, random_state=42, class_weight="balanced")
rf_tuned.fit(X_train, y_train)
y_pred_rf_tuned = rf_tuned.predict(X_test)
print(f"Random Forest Classifier (Tuned) Accuracy: {accuracy_score(y_test, y_pred_rf_tuned):.4f}")
print("\nClassification Report (Tuned):")
print(classification_report(y_test, y_pred_rf_tuned))
print("\nConfusion Matrix (Tuned):")
print(confusion_matrix(y_test, y_pred_rf_tuned))


Approach 3 : Feature Importance Analysis

In [ ]:
features_selected = pd.DataFrame({'Feature': x.columns, 'Importance': rf.feature_importances_})
features_selected = features_selected.sort_values(by='Importance', ascending=False)
features_selected

In [ ]:
x_selected = x.drop(columns = ["Phone Service_Yes", "Multiple Lines_No phone service"])

In [ ]:
X_train_selected, X_test_selected, y_train_selected, y_test_selected = train_test_split(x_selected, y, test_size=0.2, random_state=42)

In [ ]:
rf_selected = RandomForestClassifier(n_estimators=300, max_depth = 10, random_state=42, class_weight="balanced")
rf_selected.fit(X_train_selected, y_train_selected)
y_pred_rf_selected = rf_selected.predict(X_test_selected)
print(f"Random Forest Classifier (Selected Features) Accuracy: {accuracy_score(y_test_selected, y_pred_rf_selected):.4f}")
print("\nClassification Report (Selected Features):")
print(classification_report(y_test_selected, y_pred_rf_selected))

Approach 2 Combination of trees and depths

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

In [ ]:
n_estimators = [100, 200, 300, 400, 500]
max_depths = [5, 10, 15, 20]
result = []
for n in n_estimators:
    for d in max_depths:
        rf = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42, class_weight="balanced")
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        result.append({'n_estimators': n, 'max_depth': d, 'accuracy': accuracy, 'recall': recall, 'precision': precision, 'f1_score': f1})

result_df = pd.DataFrame(result)
result_df = result_df.sort_values(by=['recall','accuracy'], ascending=False)
result_df

- Hyperparameter tuning showed that both `(n_estimators=300, max_depth=10)` and `(n_estimators=400, max_depth=10)` produced identical performance metrics.

- Since `(n_estimators=300, max_depth=10)` achieved the same results with fewer trees, the `rf_tuned` model was selected as the final Random Forest model.

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
cv_accuracy = cross_val_score(rf_tuned, x, y, cv=5, scoring='accuracy')
cv_recall = cross_val_score(rf_tuned, x, y, cv=5, scoring='recall')

In [ ]:
cv_accuracy.mean()

In [ ]:
cv_recall.mean()

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
y_proba = rf_tuned.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, y_proba)


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
# plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

Customer Segmentation

In [ ]:
churn_probability = rf_tuned.predict_proba(x)[:, 1]

In [ ]:
segmentation_data = pd.DataFrame({
    'Tenure Months': df['Tenure Months'],
    'Monthly Charges': df['Monthly Charges'],
    'Total Charges': df['Total Charges'],
    'Churn Probability' : churn_probability
})

In [ ]:
segmentation_data

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
scaled_data = scaler.fit_transform(segmentation_data)

In [ ]:
from sklearn.cluster import KMeans

wcss = []
k = range(1, 16)
for i in k:
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(scaled_data)
    wcss.append(kmeans.inertia_)

sns.lineplot(x=k, y=wcss, marker = 'o')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.title('Elbow Method')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, init='k-means++', random_state=42)
clusters = kmeans.fit_predict(scaled_data)

In [ ]:
segmentation_data["Cluster"] = clusters
segmentation_data

In [ ]:
cluster_summary = segmentation_data.groupby("Cluster").mean()
cluster_summary

In [ ]:
cluster_names = {
    0 : "Budget Loyal Customers",
    1 : "High Risk New Customers",
    2 : "Premium Loyal Customers"
}

In [ ]:
segmentation_data["Cluster Segment"] = segmentation_data["Cluster"].map(cluster_names)

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='Churn Probability', y='Monthly Charges', hue='Cluster Segment', data=segmentation_data, palette='rainbow')
plt.xlabel('Churn Probability')
plt.ylabel('Monthly Charges')
plt.title('Customer Segmentation')
plt.legend(title='Cluster Segment')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='Churn Probability', y='Total Charges', hue='Cluster Segment', data=segmentation_data, palette='rainbow')
plt.xlabel('Churn Probability')
plt.ylabel('Total Charges')
plt.title('Customer Segmentation')
plt.legend(title='Cluster Segment')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='Churn Probability', y='Tenure Months', hue='Cluster Segment', data=segmentation_data, palette='rainbow')
plt.xlabel('Churn Probability')
plt.ylabel('Tenure Months')
plt.title('Customer Segmentation')
plt.legend(title='Cluster Segment')
plt.show()